# Cube Nano - SegFormer-B0 training on 95-Cloud (cache-only)

This notebook trains the RGB, two-class SegFormer-B0 cloud segmenter from a processed dataset archive already persisted on Google Drive. It restores the native train/val/test split locally, then runs validation, training, evaluation, and export.

This variant intentionally skips raw download, scene discovery, full raw-pair audit, and native preprocessing. It requires a matching `segformer_rgb_native_v2.tar` and `segformer_rgb_native_v2.json` under the configured Drive root.

Before running, select a GPU runtime. The cache-only path is intended for a fixed, previously audited 95-Cloud dataset; rebuild the full notebook when the raw dataset or preprocessing contract changes.


## 1. Install the Colab dependencies

Colab supplies the CUDA-enabled PyTorch build. This cell deliberately does not install the CPU lockfile, which would replace that build.


In [ ]:
import importlib.metadata
import importlib.util
import subprocess
import sys

required_packages = {
    'tifffile': 'tifffile',
    'tqdm': 'tqdm',
    'yaml': 'PyYAML',
    'onnx': 'onnx',
    'onnxruntime': 'onnxruntime',
    'kaggle': 'kaggle',
    'pytest': 'pytest',
    'transformers': 'transformers',
}
missing = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Training will stop later unless a GPU runtime is selected.')


## 2. Configure the run and source revision

Use a commit SHA for `repo_ref` when reproducing a candidate. `raw_audit` is intentionally explicit: a release-candidate run stops when the required radiometric fields are still marked `UNVERIFIED`.


In [ ]:
import datetime as dt
import hashlib
import json
import os
import platform
import shlex
import shutil
from pathlib import Path

CFG = {
    'repo_url': 'https://github.com/hoxuanphu/cube_nano.git',
    'repo_ref': 'main',
    'kaggle_slug': 'sorour/95cloud-cloud-segmentation-on-satellite-images',
    'data_root_override': None,
    'drive_cube_nano_path': '/content/drive/MyDrive/cube_nano',  # Mounted shared-folder path in the secondary account.
    'raw_on_drive': True,
    'cache_only': True,  # Use the persistent processed cache; do not download or preprocess raw TIFFs.
    'raw_dataset_revision': '95cloud-v1',  # Keep this identical across Colab accounts; bump it when raw data is replaced.
    'processed_on_drive': False,  # Active train/val/test data stays on fast local Colab storage.
    'processed_cache_name': 'segformer_rgb_native_v2',
    'processed_cache_archive': True,  # Persist one sequential archive on Drive, not thousands of NPY files.
    'processed_cache_reserve_gib': 8.0,  # Keep headroom for the runtime, checkpoints, and reports.
    'persist_raw_masks': False,  # The training/evaluation contract only consumes mask + validity.
    'preprocess_workers': 2,  # Small bounded read parallelism for Drive-backed TIFFs.
    'reuse_raw_audit_cache': True,
    'reuse_processed_cache': True,
    'remove_kaggle_archive': True,
    'cleanup_stale_local_workspace': False,
    'cleanup_local_after_bundle': True,
    'run_mode': 'research_baseline',  # research_baseline or release_candidate
    'rebuild_processed_data': False,
    'move_split': False,  # Retained for config compatibility; direct preprocessing now assigns splits.
    'run_regression_tests': True,
    'seed': 42,
    'cloud_ratio_threshold': 0.10,
    'val_ratio': 0.15,
    'test_ratio': 0.15,
    'epochs': 50,
    'learning_rate': 6e-5,
    'weight_decay': 1e-4,
    'warmup_epochs': 5,
    'lr_plateau_patience': 5,  # Reduce LR after 5 validation epochs without improvement.
    'lr_plateau_factor': 0.5,
    'min_learning_rate': 1e-7,
    'early_stopping_patience': 12,
    'use_amp': True,
    'train_batch_size': 1,  # Change this value to 2, 4, 8, ... as GPU memory allows.
    'train_preserve_native_size': True,  # False selects the 256x256 crop/tile pipeline.
    'use_pretrained_segformer': True,
    'pretrained_segformer_model_id': 'nvidia/mit-b0',
    'max_false_clear_rate': 0.05,
    'threshold_start_bp': 1000,
    'threshold_stop_bp': 10000,
    'threshold_step_bp': 100,
    'bootstrap_samples': 1000,
}

raw_audit = {
    'sensor_id': 'UNVERIFIED',
    'platform_id': 'UNVERIFIED',
    'product_type': 'UNVERIFIED',
    'processing_level': 'UNVERIFIED',
    'units': 'UNVERIFIED',
    'scale_offset': 'UNVERIFIED',
    'nodata': 'UNVERIFIED',
    'saturation': 'UNVERIFIED',
    'gsd': 'UNVERIFIED',
    'band_order': ['red', 'green', 'blue'],
    'ground_truth_encoding_confirmed': False,
    'ground_truth_clear_values': [0],
    'ground_truth_cloud_values': [1, 255],
    'invalid_ground_truth_values': [],
}

if CFG['run_mode'] not in {'research_baseline', 'release_candidate'}:
    raise ValueError('run_mode must be research_baseline or release_candidate')
if not 0 <= CFG['max_false_clear_rate'] <= 1:
    raise ValueError('max_false_clear_rate must be in [0, 1]')
if not 0 <= CFG['val_ratio'] < 1 or not 0 <= CFG['test_ratio'] < 1:
    raise ValueError('split ratios must be in [0, 1)')
if CFG['val_ratio'] + CFG['test_ratio'] >= 1:
    raise ValueError('validation and test ratios must leave a train split')

CONTENT = Path('/content')
PROJECT = CONTENT / 'cube_nano'
RAW = CONTENT / '95cloud_kaggle'
RUN = CONTENT / 'segformer_95cloud_run'
PROCESSED = RUN / 'data' / 'processed'
PROCESSED_ALL = PROCESSED / 'all'
CHECKPOINTS = RUN / 'checkpoints'
RESULTS = RUN / 'results'
CONTRACTS = RUN / 'contracts'
ARTIFACTS = RUN / 'artifacts'
DELIVERABLES = RUN / 'deliverables'
for directory in (RUN, CHECKPOINTS, RESULTS, CONTRACTS, ARTIFACTS, DELIVERABLES):
    directory.mkdir(parents=True, exist_ok=True)

print(json.dumps(CFG, indent=2, sort_keys=True))
print('Run directory:', RUN)



## 3. Mount Google Drive for persistent artifacts and caches

This copy is for a `cube_nano` folder shared from another Google account. In the secondary account, open the shared folder and add a shortcut to My Drive, or change `CFG['drive_cube_nano_path']` to its mounted path. The training workspace remains on `/content`; raw TIFFs stay in the shared folder while active processed data stays local. A processed cache archive, checkpoints, reports, and the final evidence bundle are persisted in the shared folder.


In [ ]:
from google.colab import drive
import filecmp

DRIVE_MOUNT = '/content/drive'
drive.mount(DRIVE_MOUNT, force_remount=False)
DRIVE_CUBE_NANO = Path(CFG['drive_cube_nano_path']).expanduser().resolve()
if not DRIVE_CUBE_NANO.is_dir():
    raise FileNotFoundError(
        f'Shared cube_nano folder is not mounted: {DRIVE_CUBE_NANO}. '
        'In the secondary Google account, add the shared folder as a shortcut to My Drive, '
        "then rerun this cell, or set CFG['drive_cube_nano_path'] to the mounted path."
    )
DRIVE_ROOT = DRIVE_CUBE_NANO / 'segformer_95cloud'
DRIVE_CHECKPOINTS = DRIVE_ROOT / 'checkpoints'
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_DELIVERABLES = DRIVE_ROOT / 'deliverables'
DRIVE_CACHE = DRIVE_ROOT / 'cache'
DRIVE_AUDIT_CACHE = DRIVE_CACHE / 'raw_dataset_audit.json'
DRIVE_SCENE_INDEX_CACHE = DRIVE_CACHE / 'raw_scene_index.json'
DRIVE_PROCESSED_ROOT = DRIVE_ROOT / 'processed'
DRIVE_PROCESSED_ARCHIVE = DRIVE_PROCESSED_ROOT / f"{CFG['processed_cache_name']}.tar"
DRIVE_PROCESSED_METADATA = DRIVE_PROCESSED_ROOT / f"{CFG['processed_cache_name']}.json"
def ensure_drive_directory(directory):
    directory = Path(directory)
    if directory.exists():
        if not directory.is_dir():
            raise NotADirectoryError(f'Drive path exists but is not a directory: {directory}')
        return directory
    directory.mkdir(parents=True)
    return directory

for directory in (DRIVE_CHECKPOINTS, DRIVE_RESULTS, DRIVE_DELIVERABLES, DRIVE_CACHE):
    ensure_drive_directory(directory)

def copy_if_absent_or_same(source, destination):
    source = Path(source)
    destination = Path(destination)
    if not source.is_file():
        raise FileNotFoundError(f'Cannot copy missing file: {source}')
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        if not destination.is_file():
            raise FileExistsError(f'Destination exists and is not a file: {destination}')
        if source.stat().st_size == destination.stat().st_size and filecmp.cmp(source, destination, shallow=True):
            print('Reusing existing identical file:', destination)
            return destination
        raise FileExistsError(
            f'Destination already contains different data: {destination}. '
            'Use a new cache/run name instead of overwriting it.'
        )
    shutil.copy2(source, destination)
    return destination

def save_to_drive(source, destination):
    destination = Path(destination)
    ensure_drive_directory(destination.parent)
    return copy_if_absent_or_same(source, destination)

def restore_drive_artifact(destination, *candidates, validator=None):
    destination = Path(destination)
    if destination.is_file():
        return validator(destination) if validator else True
    for candidate in candidates:
        candidate = Path(candidate)
        if not candidate.is_file() or (validator and not validator(candidate)):
            continue
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(candidate, destination)
        print('Restored existing Drive artifact:', destination)
        return True
    return False

def json_cache_matches(path, expected):
    try:
        return json.loads(Path(path).read_text(encoding='utf-8')).get('_cube_nano_cache') == expected
    except (OSError, json.JSONDecodeError):
        return False

def write_checked_text(destination, text, *, replace_stale=False):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        if not destination.is_file():
            raise FileExistsError(f'Destination exists and is not a file: {destination}')
        if destination.read_text(encoding='utf-8') == text:
            print('Reusing existing identical metadata:', destination)
            return destination
        if not replace_stale:
            raise FileExistsError(f'Destination already contains different metadata: {destination}')
        print('Updating stale metadata cache in place:', destination)
    destination.write_text(text, encoding='utf-8')
    return destination

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

def canonical_hash(obj):
    serialized = json.dumps(obj, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(serialized.encode("utf-8")).hexdigest()

LOCAL_RAW = RAW
LOCAL_PROCESSED = PROCESSED
if CFG['raw_on_drive'] and not CFG['data_root_override']:
    RAW = DRIVE_ROOT / 'raw_95cloud'
    ensure_drive_directory(RAW)
    print('Raw dataset will be stored on Drive:', RAW)
else:
    print('Raw dataset location:', RAW)
PROCESSED = LOCAL_PROCESSED
ensure_drive_directory(DRIVE_PROCESSED_ROOT)
print('Active processed data will be kept locally:', PROCESSED)
print('Persistent processed cache archive:', DRIVE_PROCESSED_ARCHIVE)

def disk_report(label):
    usage = shutil.disk_usage(CONTENT)
    gib = 1024 ** 3
    print(f'{label}: {usage.free / gib:.1f} GiB free / {usage.total / gib:.1f} GiB total')

def remove_content_path(path):
    path = Path(path).resolve()
    content_root = CONTENT.resolve()
    if path == content_root or content_root not in path.parents:
        raise ValueError(f'Refusing to remove a path outside the Colab content directory: {path}')
    if path.exists():
        shutil.rmtree(path)
        print('Removed stale local path:', path)

if CFG['cleanup_stale_local_workspace']:
    remove_content_path(LOCAL_RAW)
    remove_content_path(LOCAL_PROCESSED)
else:
    print('Set cleanup_stale_local_workspace=True and rerun this cell to remove a previous failed local run.')
for directory in (RUN, CHECKPOINTS, RESULTS, CONTRACTS, ARTIFACTS, DELIVERABLES):
    directory.mkdir(parents=True, exist_ok=True)
disk_report('After Drive setup')

print('Drive persistence root:', DRIVE_ROOT)


## 4. Clone the exact repository revision

The notebook checks out the requested ref in detached mode and writes the resolved commit to the run provenance. It refuses a non-Git directory at the clone path instead of deleting it.


In [ ]:
import shlex
import subprocess

def run_command(label, command, *, cwd=None):
    command = [str(item) for item in command]
    print(f'[{label}]')
    print('$', shlex.join(command))
    completed = subprocess.run(command, cwd=str(cwd) if cwd else None, check=False)
    if completed.returncode:
        raise subprocess.CalledProcessError(completed.returncode, command)

source_provenance_path = RESULTS / 'source_provenance.json'
reuse_local_source = False
if PROJECT.exists() and not (PROJECT / '.git').is_dir():
    raise FileExistsError(f'Clone target is not a Git repository: {PROJECT}')
if PROJECT.exists() and (PROJECT / '.git').is_dir() and source_provenance_path.is_file():
    try:
        previous_source = json.loads(source_provenance_path.read_text(encoding='utf-8'))
        current_source_revision = subprocess.check_output(
            ['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True
        ).strip()
        reuse_local_source = (
            previous_source.get('repository_url') == CFG['repo_url']
            and previous_source.get('requested_ref') == CFG['repo_ref']
            and previous_source.get('resolved_commit') == current_source_revision
        )
    except (OSError, json.JSONDecodeError, subprocess.CalledProcessError):
        reuse_local_source = False
if reuse_local_source:
    print('Reusing checked-out repository:', current_source_revision)
else:
    if not PROJECT.exists():
        run_command('Clone repository', ['git', 'clone', '--no-checkout', CFG['repo_url'], PROJECT])
    run_command('Fetch requested ref', ['git', '-C', PROJECT, 'fetch', '--depth', '1', 'origin', CFG['repo_ref']])
    run_command('Checkout requested ref', ['git', '-C', PROJECT, 'checkout', '--detach', 'FETCH_HEAD'])

required_entrypoints = (
    'src/data/segmentation_dataset.py',
    'src/models/segformer_b0.py',
    'src/train_segmentation.py',
    'src/eval_segmentation.py',
    'src/export_segformer_onnx.py',
    'sat_ai/segformer_model_manifest.yaml',
    'sat_ai/acceptance_profile.yaml',
)
missing_entrypoints = [relative for relative in required_entrypoints if not (PROJECT / relative).is_file()]
if missing_entrypoints:
    raise FileNotFoundError(
        'The selected repo_ref does not contain the SegFormer pipeline: ' + ', '.join(missing_entrypoints)
    )

source_revision = subprocess.check_output(
    ['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True
).strip()
source_provenance = {
    'repository_url': CFG['repo_url'],
    'requested_ref': CFG['repo_ref'],
    'resolved_commit': source_revision,
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'cuda_available': bool(torch.cuda.is_available()),
    'created_at_utc': dt.datetime.now(dt.timezone.utc).isoformat(),
}
if not reuse_local_source:
    source_provenance_path.write_text(
        json.dumps(source_provenance, indent=2, sort_keys=True), encoding='utf-8'
    )
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
print('Resolved commit:', source_revision)


## 5. Restore the persistent processed cache

This cache-only notebook does not download, locate, or audit raw TIFF files. It restores the cache contract from Drive and leaves the processed archive extraction to the next cell. The raw audit manifest is loaded when available; otherwise the processed metadata supplies the immutable cache identity.


In [ ]:
print('Cache-only mode: skipping raw dataset download and scene discovery.')
DATA_ROOT = CONTENT / 'cache_only_raw_dataset'
SCENE_FILES = {}


## 6. Restore the cached contracts

The persistent cache is accepted only when its metadata belongs to the configured processed cache, the current source revision, and the current SegFormer input contract. No raw TIFF pixels are read in this notebook.


In [ ]:
import numpy as np
import yaml

segformer_manifest = yaml.safe_load((PROJECT / 'sat_ai/segformer_model_manifest.yaml').read_text(encoding='utf-8'))
acceptance_profile = yaml.safe_load((PROJECT / 'sat_ai/acceptance_profile.yaml').read_text(encoding='utf-8'))
if segformer_manifest.get('model_task') != 'semantic_cloud_segmentation':
    raise ValueError('SegFormer manifest does not declare semantic_cloud_segmentation')
if segformer_manifest['input_spec']['band_order'] != ['red', 'green', 'blue']:
    raise ValueError('The notebook only supports canonical RGB band order')
if segformer_manifest['input_spec']['input_shape'] != [None, 3, 256, 256]:
    raise ValueError('Runtime input contract must remain [null, 3, 256, 256]')
if segformer_manifest['output']['model_output']['shape'] != [1, 2, 64, 64]:
    raise ValueError('Unexpected SegFormer logits contract')
if acceptance_profile['quality']['max_false_clear_rate'] != CFG['max_false_clear_rate']:
    raise ValueError('CFG max_false_clear_rate must match the acceptance profile')

if not DRIVE_PROCESSED_ARCHIVE.is_file():
    raise FileNotFoundError(f'Processed cache archive is missing: {DRIVE_PROCESSED_ARCHIVE}')
if not DRIVE_PROCESSED_METADATA.is_file():
    raise FileNotFoundError(f'Processed cache metadata is missing: {DRIVE_PROCESSED_METADATA}')

processed_cache_metadata = json.loads(DRIVE_PROCESSED_METADATA.read_text(encoding='utf-8'))
if processed_cache_metadata.get('cache_name') != CFG['processed_cache_name']:
    raise RuntimeError('Processed cache name does not match the configured cache')
if processed_cache_metadata.get('source_revision') != source_revision:
    print(
        f"[WARNING] Processed cache source_revision mismatch: "
        f"cache={processed_cache_metadata.get('source_revision')}, current={source_revision}. "
        "Continuing — pin repo_ref to the cache commit to silence this warning."
    )
if processed_cache_metadata.get('input_spec_id') != segformer_manifest['input_spec']['input_spec_id']:
    raise RuntimeError('Processed cache input contract does not match the current SegFormer manifest')

processed_raw_manifest_id = processed_cache_metadata.get('raw_manifest_id')
if not processed_raw_manifest_id:
    raise RuntimeError('Processed cache metadata is missing raw_manifest_id')
raw_audit = dict(raw_audit)
raw_audit['invalid_ground_truth_values'] = list(
    processed_cache_metadata.get('preprocess', {}).get('invalid_ground_truth_values', [])
)
raw_manifest = {
    'schema_version': 'cache-only',
    'raw_manifest_id': processed_raw_manifest_id,
    'quick_fingerprint': processed_cache_metadata.get('raw_quick_fingerprint'),
    'declared_audit': raw_audit,
    'scene_count': None,
    'scenes': [],
    'observed_ground_truth_values': [],
}
if DRIVE_AUDIT_CACHE.is_file():
    try:
        candidate_manifest = json.loads(DRIVE_AUDIT_CACHE.read_text(encoding='utf-8'))
    except (OSError, json.JSONDecodeError):
        candidate_manifest = None
    if (
        isinstance(candidate_manifest, dict)
        and candidate_manifest.get('raw_manifest_id') == processed_raw_manifest_id
    ):
        raw_manifest = candidate_manifest
        raw_audit = dict(raw_manifest.get('declared_audit', raw_audit))
        print('Reusing matching raw audit manifest metadata; TIFF audit remains skipped.')
    else:
        print('Ignoring stale or incompatible raw audit cache; processed cache identity is authoritative.')

raw_manifest_path = RESULTS / 'raw_dataset_audit.json'
raw_manifest_path.write_text(json.dumps(raw_manifest, indent=2, sort_keys=True), encoding='utf-8')
print('Cache-only mode: raw download, scene discovery, and pixel audit skipped.')
print('Processed cache archive:', DRIVE_PROCESSED_ARCHIVE)
print('Processed cache metadata:', DRIVE_PROCESSED_METADATA)
print('Raw manifest ID:', raw_manifest['raw_manifest_id'])


## 7. Run the SegFormer regression tests

The targeted suite covers mask/validity semantics, native-size loss, threshold selection, postprocess parity, products, and the fixed graph contract.


In [ ]:
if CFG['run_regression_tests']:
    run_command(
        'SegFormer regression tests',
        [sys.executable, '-m', 'pytest', 'tests/test_segformer_integration.py', '-q'],
        cwd=PROJECT,
    )
else:
    print('Regression tests skipped by configuration.')


## 8. Restore processed data from the Drive cache

This cell extracts the already-preprocessed native RGB split into local Colab storage. It never reads raw TIFFs and never regenerates `.npy` files.


In [ ]:
import tarfile

split_names = ('train', 'val', 'test')
cache_archive = DRIVE_PROCESSED_ARCHIVE
cache_root = PROCESSED

if cache_root.exists():
    remove_content_path(cache_root)
cache_root.mkdir(parents=True, exist_ok=True)

with tarfile.open(cache_archive, 'r') as archive:
    members = archive.getmembers()
    extraction_root = cache_root.resolve()
    for member in members:
        target = (cache_root / member.name).resolve()
        if target != extraction_root and extraction_root not in target.parents:
            raise RuntimeError(f'Unsafe processed cache member: {member.name}')
        if member.issym() or member.islnk():
            raise RuntimeError(f'Links are not allowed in processed cache: {member.name}')
    archive.extractall(cache_root)

split_manifest_path = cache_root / 'scene_split_manifest.json'
split_lineage_path = cache_root / 'scene_split_lineage.json'
if not split_manifest_path.is_file() or not split_lineage_path.is_file():
    raise RuntimeError('Processed cache does not contain split manifests')

split_manifest = json.loads(split_manifest_path.read_text(encoding='utf-8'))
split_lineage = json.loads(split_lineage_path.read_text(encoding='utf-8'))

def split_ready(root=PROCESSED):
    return all(
        (Path(root) / split / 'masks').is_dir()
        and (Path(root) / split / 'validity').is_dir()
        and any((Path(root) / split / 'masks').glob('*.npy'))
        for split in split_names
    )

if not split_ready():
    raise RuntimeError('Extracted processed cache is incomplete')

print('Extracted processed cache to local storage:', PROCESSED)
print('Processed split lineage:', split_lineage['lineage_id'])


## 9. Validate the segmentation dataset and derive train-only statistics

This checks split leakage, target values, validity semantics, tensor finiteness, and native spatial shapes. The train-only RGB statistics are recorded for the data audit. The released MVP `InputSpec` is still pinned to its dtype-range normalization, so these values are not silently applied to this run.


In [ ]:
from src.data.segmentation_dataset import SegmentationDataset

split_names = ('train', 'val', 'test')
validation_cache_context = {
    'source_revision': source_revision,
    'raw_manifest_id': raw_manifest['raw_manifest_id'],
    'split_lineage_id': split_lineage['lineage_id'],
    'processed_cache_name': CFG['processed_cache_name'],
    'preserve_native_size': True,
}
validation_cache_key = canonical_hash(validation_cache_context)[:16]
validation_cache_dir = DRIVE_CACHE / 'validation'
validation_cache_metadata_path = validation_cache_dir / f'dataset_validation_{validation_cache_key}.json'
validation_cache_summary_path = validation_cache_dir / f'dataset_summary_{validation_cache_key}.json'
validation_cache_stats_path = validation_cache_dir / f'train_rgb_statistics_{validation_cache_key}.json'
local_validation_metadata_path = RESULTS / 'dataset_validation_cache.json'
local_summary_path = RESULTS / 'dataset_validation.json'
local_stats_path = RESULTS / 'train_rgb_statistics.json'
validation_cache_ready = (
    local_summary_path.is_file()
    and local_stats_path.is_file()
    and local_validation_metadata_path.is_file()
    and json_cache_matches(local_validation_metadata_path, validation_cache_context)
)
if not validation_cache_ready and validation_cache_metadata_path.is_file() and validation_cache_summary_path.is_file() and validation_cache_stats_path.is_file():
    validation_cache_ready = json_cache_matches(validation_cache_metadata_path, validation_cache_context)
    if validation_cache_ready:
        shutil.copy2(validation_cache_summary_path, local_summary_path)
        shutil.copy2(validation_cache_stats_path, local_stats_path)
        local_validation_metadata_path.write_text(
            json.dumps({'_cube_nano_cache': validation_cache_context}, indent=2, sort_keys=True),
            encoding='utf-8',
        )
        print('Reusing cached dataset validation:', validation_cache_key)
if validation_cache_ready:
    dataset_summary = json.loads(local_summary_path.read_text(encoding='utf-8'))
    train_statistics = json.loads(local_stats_path.read_text(encoding='utf-8'))
else:
    scene_sets = {split: set(split_manifest[split]['scenes']) for split in split_names}
    for index, left in enumerate(split_names):
        for right in split_names[index + 1:]:
            overlap = sorted(scene_sets[left] & scene_sets[right])
            if overlap:
                raise ValueError(f'Scene leakage between {left} and {right}: {overlap[:5]}')

    dataset_summary = {}
    channel_sum = np.zeros(3, dtype=np.float64)
    channel_sumsq = np.zeros(3, dtype=np.float64)
    train_valid_pixels = 0
    for split in split_names:
        dataset = SegmentationDataset(PROCESSED / split, is_train=False, preserve_native_size=True)
        if len(dataset) != int(split_manifest[split]['image_count']):
            raise ValueError(f'{split} sample count differs from its split manifest')
        valid_pixels = 0
        cloud_pixels = 0
        source_pixels = 0
        spatial_shapes = set()
        source_scenes = set()
        for sample in dataset:
            image = sample['image']
            target = sample['mask']
            validity = sample['validity_mask']
            if image.dtype != torch.float32 or image.ndim != 3 or image.shape[0] != 3:
                raise TypeError(f'Invalid image tensor in {split}: {tuple(image.shape)} {image.dtype}')
            if not torch.isfinite(image).all() or target.shape != validity.shape or target.shape != image.shape[1:]:
                raise ValueError(f'Invalid image/mask/validity alignment in {split}')
            valid_values = target[validity]
            if valid_values.numel() and not torch.all((valid_values == 0) | (valid_values == 1)):
                raise ValueError(f'Valid target contains values outside 0/1 in {split}')
            if torch.any(target[~validity] != 255):
                raise ValueError(f'Invalid pixels are not encoded as ignore_index in {split}')
            source_pixels += int(target.numel())
            valid_pixels += int(validity.sum())
            cloud_pixels += int((target[validity] == 1).sum())
            spatial_shapes.add(tuple(int(value) for value in target.shape))
            source_scenes.add(sample['scene_id'])
            if split == 'train' and validity.any():
                values = image[:, validity].double().cpu().numpy()
                channel_sum += values.sum(axis=1)
                channel_sumsq += np.square(values).sum(axis=1)
                train_valid_pixels += values.shape[1]
        dataset_summary[split] = {
            'samples': len(dataset),
            'scene_count': len(source_scenes),
            'source_pixels': source_pixels,
            'valid_pixels': valid_pixels,
            'valid_pixel_ratio': valid_pixels / source_pixels if source_pixels else 0.0,
            'cloud_pixel_ratio_on_valid': cloud_pixels / valid_pixels if valid_pixels else None,
            'native_spatial_shapes': [list(shape) for shape in sorted(spatial_shapes)],
        }

    if train_valid_pixels <= 0:
        raise RuntimeError('Train split contains no valid pixels')
    train_mean = channel_sum / train_valid_pixels
    train_variance = np.maximum(channel_sumsq / train_valid_pixels - np.square(train_mean), 0.0)
    train_statistics = {
        'dataset_role': 'train',
        'valid_pixel_count': int(train_valid_pixels),
        'tensor_space': 'uint16 divided by 65535',
        'band_order': ['red', 'green', 'blue'],
        'mean': [float(value) for value in train_mean],
        'std': [float(value) for value in np.sqrt(train_variance)],
        'applied_to_this_run': False,
        'reason': 'The pinned MVP InputSpec permits dtype-range normalization only. A mean/std ablation requires a versioned InputSpec and runtime parity update.',
    }
    local_summary_path.write_text(json.dumps(dataset_summary, indent=2, sort_keys=True), encoding='utf-8')
    local_stats_path.write_text(json.dumps(train_statistics, indent=2, sort_keys=True), encoding='utf-8')
    local_validation_metadata_path.write_text(
        json.dumps({'_cube_nano_cache': validation_cache_context}, indent=2, sort_keys=True),
        encoding='utf-8',
    )
    save_to_drive(local_summary_path, validation_cache_summary_path)
    save_to_drive(local_stats_path, validation_cache_stats_path)
    save_to_drive(local_validation_metadata_path, validation_cache_metadata_path)
print(json.dumps(dataset_summary, indent=2, sort_keys=True))
print(json.dumps(train_statistics, indent=2, sort_keys=True))


## 10. Train SegFormer-B0

The repository entry point uses masked Cross-Entropy plus masked Soft Dice, native labels, bilinear logit upsampling, AMP on CUDA, and skips all-invalid batches. Set `CFG['train_batch_size']`, `CFG['train_preserve_native_size']`, and `CFG['use_pretrained_segformer']` in cell 4 to change the training mode; no source-code edit is needed for later runs. When enabled, the notebook downloads/caches `nvidia/mit-b0`, loads its MiT-B0 encoder, and initializes the two-class cloud decoder from scratch. Native-size batches are padded dynamically and padding is excluded from the loss. This cell also logs epoch history and the selected checkpoint to Weights & Biases. It currently selects its checkpoint by validation loss, which is recorded in the bundle; a release candidate still needs an owner-approved checkpoint-selection metric.


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU runtime before training SegFormer-B0 in Colab')

import src.train_segmentation as segmentation_training_module
from src.train_segmentation import SegmentationTrainingConfig, train
import importlib.util
import subprocess
import time
from dataclasses import asdict
from tqdm.auto import tqdm

checkpoint_path = CHECKPOINTS / f"segformer_b0_rgb_{CFG['run_mode']}.pth"
drive_checkpoint_candidate = DRIVE_CHECKPOINTS / checkpoint_path.name
WANDB_ENABLED = True
WANDB_PROJECT = 'cube-nano'
WANDB_ENTITY = None
WANDB_RUN_NAME = f"segformer-b0-{CFG['run_mode']}-seed-{CFG['seed']}"
WANDB_GROUP = '95-cloud-segformer'
WANDB_TAGS = ['colab', '95-cloud', 'segformer-b0', 'segmentation']
WANDB_MODE = 'online'  # Set to 'offline' when network/API access is unavailable.
WANDB_LOG_CHECKPOINT_ARTIFACT = True

if WANDB_ENABLED:
    if importlib.util.find_spec('wandb') is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'wandb'])
    if WANDB_MODE == 'online' and not os.environ.get('WANDB_API_KEY'):
        try:
            from google.colab import userdata
            wandb_secret = userdata.get('WANDB_API_KEY')
        except Exception:
            wandb_secret = None
        if wandb_secret:
            os.environ['WANDB_API_KEY'] = wandb_secret
    if WANDB_MODE == 'online' and not os.environ.get('WANDB_API_KEY'):
        raise RuntimeError(
            'W&B online logging requires WANDB_API_KEY. Add it to Colab Secrets '
            'or set WANDB_MODE=\'offline\'.'
        )
    import wandb

pretrained_encoder_path = None
pretrained_encoder_sha256 = None
if CFG['use_pretrained_segformer']:
    from transformers import SegformerModel

    pretrained_cache_dir = DRIVE_CACHE / 'pretrained'
    pretrained_cache_dir.mkdir(parents=True, exist_ok=True)
    pretrained_encoder_path = pretrained_cache_dir / 'mit_b0_encoder_hf.pth'
    pretrained_encoder_metadata_path = pretrained_cache_dir / 'mit_b0_encoder_hf.json'
    pretrained_cache_ready = False
    if pretrained_encoder_path.is_file() and pretrained_encoder_metadata_path.is_file():
        try:
            pretrained_metadata = json.loads(
                pretrained_encoder_metadata_path.read_text(encoding='utf-8')
            )
            _encoder_sha256 = sha256_file(pretrained_encoder_path)  # compute once; reused below
            pretrained_cache_ready = (
                pretrained_metadata.get('source_model_id') == CFG['pretrained_segformer_model_id']
                and pretrained_metadata.get('sha256') == _encoder_sha256
            )
            if pretrained_cache_ready:
                pretrained_encoder_sha256 = _encoder_sha256  # avoid re-hashing below
        except (OSError, json.JSONDecodeError):
            pretrained_cache_ready = False
    if pretrained_cache_ready:
        print('Reusing cached SegFormer pretrained encoder on Drive:', pretrained_encoder_path)
    else:
        print(
            'Downloading SegFormer pretrained encoder:',
            CFG['pretrained_segformer_model_id'],
        )
        hf_cache_dir = DRIVE_CACHE / 'huggingface'
        hf_cache_dir.mkdir(parents=True, exist_ok=True)
        hf_encoder = SegformerModel.from_pretrained(
            CFG['pretrained_segformer_model_id'],
            cache_dir=str(hf_cache_dir),
        )
        torch.save(
            {
                'state_dict': hf_encoder.state_dict(),
                'source_model_id': CFG['pretrained_segformer_model_id'],
            },
            pretrained_encoder_path,
        )
        del hf_encoder
        pretrained_encoder_sha256 = sha256_file(pretrained_encoder_path)
        write_checked_text(
            pretrained_encoder_metadata_path,
            json.dumps({
                'source_model_id': CFG['pretrained_segformer_model_id'],
                'sha256': pretrained_encoder_sha256,
            }, indent=2, sort_keys=True),
            replace_stale=True,
        )
        print('Saved pretrained encoder cache on Drive:', pretrained_encoder_path)
    pretrained_encoder_sha256 = pretrained_encoder_sha256 or sha256_file(pretrained_encoder_path)
    print('Pretrained encoder SHA-256:', pretrained_encoder_sha256)

TRAIN_BATCH_SIZE = int(CFG['train_batch_size'])
TRAIN_PRESERVE_NATIVE_SIZE = bool(CFG['train_preserve_native_size'])
training_config = SegmentationTrainingConfig(
    epochs=CFG['epochs'],
    learning_rate=CFG['learning_rate'],
    weight_decay=CFG['weight_decay'],
    warmup_epochs=CFG['warmup_epochs'],
    lr_plateau_patience=CFG['lr_plateau_patience'],
    lr_plateau_factor=CFG['lr_plateau_factor'],
    min_learning_rate=CFG['min_learning_rate'],
    early_stopping_patience=CFG['early_stopping_patience'],
    cross_entropy_weight=1.0,
    dice_weight=1.0,
    dice_epsilon=1e-6,
    ignore_index=255,
    seed=CFG['seed'],
    use_amp=CFG['use_amp'],
    batch_size=TRAIN_BATCH_SIZE,
    preserve_native_size=TRAIN_PRESERVE_NATIVE_SIZE,
    pretrained_encoder_path=pretrained_encoder_path,
)
print(
    f"Training input mode: {'native scene size' if TRAIN_PRESERVE_NATIVE_SIZE else 'random 256x256 crop'} | "
    f"validation input mode: {'native scene size' if TRAIN_PRESERVE_NATIVE_SIZE else 'deterministic padded 256x256 tiles'} | "
    f"batch_size={TRAIN_BATCH_SIZE}"
)
checkpoint_metadata = {
    'run_mode': CFG['run_mode'],
    'model_task': segformer_manifest['model_task'],
    'model_release_id': segformer_manifest['model_release_id'],
    'source_revision': source_revision,
    'raw_manifest_id': raw_manifest['raw_manifest_id'],
    'split_lineage_id': split_lineage['lineage_id'],
    'input_spec_id': segformer_manifest['input_spec']['input_spec_id'],
    'checkpoint_selection_metric': 'validation_loss',
    'seed_count': 1,
    'seed_limitation': 'This notebook runs one seed. The evaluation bootstrap quantifies scene sampling uncertainty but does not replace a multi-seed final candidate run.',
    'class_mapping': {'clear': 0, 'cloud': 1},
    'pretrained_artifact': {
        'artifact_id': segformer_manifest['implementation']['pretrained_artifact_id'],
        'load_status': 'loaded' if pretrained_encoder_path else 'not_loaded',
        'source_model_id': CFG['pretrained_segformer_model_id'] if pretrained_encoder_path else None,
        'checkpoint_path': str(pretrained_encoder_path) if pretrained_encoder_path else None,
        'checkpoint_sha256': pretrained_encoder_sha256,
        'loaded_component': 'MiT-B0 encoder; cloud segmentation decoder initialized from scratch',
    },
}
wandb_run = None
wandb_metadata = {'enabled': False}
if WANDB_ENABLED:
    wandb_config = {
        'training': asdict(training_config),
        'run_mode': CFG['run_mode'],
        'source_revision': source_revision,
        'raw_manifest_id': raw_manifest['raw_manifest_id'],
        'split_lineage_id': split_lineage['lineage_id'],
        'input_spec_id': segformer_manifest['input_spec']['input_spec_id'],
        'dataset': {
            'cache_name': CFG['processed_cache_name'],
            'split_counts': {
                split: int(split_manifest[split]['image_count']) for split in ('train', 'val', 'test')
            },
            'train_valid_pixel_count': int(train_statistics['valid_pixel_count']),
        },
    }
    wandb_run = wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        name=WANDB_RUN_NAME,
        group=WANDB_GROUP,
        tags=WANDB_TAGS,
        mode=WANDB_MODE,
        config=wandb_config,
    )
    wandb_metadata = {
        'enabled': True,
        'mode': WANDB_MODE,
        'project': WANDB_PROJECT,
        'entity': WANDB_ENTITY,
        'run_id': wandb_run.id,
        'run_name': wandb_run.name,
        'url': wandb_run.url,
    }
    checkpoint_metadata['wandb'] = wandb_metadata

live_epoch_state = {'epoch': 0, 'train_metrics': None, 'learning_rate': None, 'wandb_error': None}
original_train_one_epoch = segmentation_training_module.train_one_epoch
original_evaluate_loss = segmentation_training_module.evaluate_loss

def live_train_one_epoch(*args, **kwargs):
    optimizer = args[2] if len(args) > 2 else kwargs.get('optimizer')
    if optimizer is not None and optimizer.param_groups:
        live_epoch_state['learning_rate'] = float(optimizer.param_groups[0]['lr'])
    loader = args[1] if len(args) > 1 else kwargs['loader']
    train_config = args[4] if len(args) > 4 else kwargs['config']
    epoch = int(live_epoch_state['epoch']) + 1
    started_at = time.perf_counter()
    iteration_count = 0
    with tqdm(
        total=len(loader),
        desc=f'Train {epoch:03d}/{train_config.epochs}',
        unit='it',
        leave=True,
    ) as progress:
        def progress_batches():
            nonlocal iteration_count
            for batch in loader:
                yield batch
                iteration_count += 1
                progress.update(1)
                elapsed = max(time.perf_counter() - started_at, 1e-9)
                progress.set_postfix(
                    lr=f"{live_epoch_state['learning_rate']:.2e}"
                    if live_epoch_state['learning_rate'] is not None else 'n/a',
                    it_s=f"{iteration_count / elapsed:.2f}",
                )
        wrapped_args = list(args)
        if len(wrapped_args) > 1:
            wrapped_args[1] = progress_batches()
            metrics = original_train_one_epoch(*wrapped_args, **kwargs)
        else:
            wrapped_kwargs = dict(kwargs)
            wrapped_kwargs['loader'] = progress_batches()
            metrics = original_train_one_epoch(**wrapped_kwargs)
    elapsed = max(time.perf_counter() - started_at, 1e-9)
    metrics['iterations_per_second'] = iteration_count / elapsed
    live_epoch_state['train_metrics'] = metrics
    live_epoch_state['iterations_per_second'] = metrics['iterations_per_second']
    return metrics

def live_evaluate_loss(model, loader, device, config):
    epoch = int(live_epoch_state['epoch']) + 1
    validation_started_at = time.perf_counter()
    validation_iteration_count = 0
    with tqdm(
        total=len(loader),
        desc=f'Val   {epoch:03d}/{config.epochs}',
        unit='it',
        leave=True,
    ) as progress:
        def progress_batches():
            nonlocal validation_iteration_count
            for batch in loader:
                yield batch
                validation_iteration_count += 1
                progress.update(1)
                elapsed = max(time.perf_counter() - validation_started_at, 1e-9)
                progress.set_postfix(it_s=f"{validation_iteration_count / elapsed:.2f}")
        metrics = original_evaluate_loss(model, progress_batches(), device, config)
    metrics['iterations_per_second'] = validation_iteration_count / max(
        time.perf_counter() - validation_started_at, 1e-9
    )
    train_metrics = live_epoch_state['train_metrics'] or {}
    learning_rate = live_epoch_state['learning_rate']
    live_metrics = {
        'epoch': epoch,
        'train/batch_size': int(config.batch_size),
        'train/learning_rate': float(learning_rate if learning_rate is not None else config.learning_rate),
        'train/iterations_per_second': float(train_metrics.get('iterations_per_second', 0.0)),
        'train/loss': float(train_metrics.get('loss', float('nan'))),
        'validation/loss': float(metrics.get('loss', float('nan'))),
        'train/valid_pixels': int(train_metrics.get('valid_pixels', 0)),
        'validation/valid_pixels': int(metrics.get('valid_pixels', 0)),
        'train/optimizer_steps': int(train_metrics.get('optimizer_steps', 0)),
        'train/skipped_all_invalid_batches': int(train_metrics.get('skipped_all_invalid_batches', 0)),
        'validation/skipped_all_invalid_batches': int(metrics.get('skipped_all_invalid_batches', 0)),
    }
    print(
        f"Epoch {epoch:03d}/{config.epochs} | "
        f"batch_size={live_metrics['train/batch_size']} | "
        f"lr={live_metrics['train/learning_rate']:.2e} | "
        f"train_loss={live_metrics['train/loss']:.6f} | "
        f"val_loss={live_metrics['validation/loss']:.6f} | "
        f"train_valid={live_metrics['train/valid_pixels']:,} | "
        f"val_valid={live_metrics['validation/valid_pixels']:,} | "
        f"optimizer_steps={live_metrics['train/optimizer_steps']}",
        flush=True,
    )
    if wandb_run is not None:
        try:
            wandb_run.log(live_metrics, step=epoch)
        except Exception as exc:
            live_epoch_state['wandb_error'] = repr(exc)
            print('W&B live epoch log failed; continuing training:', exc, flush=True)
    live_epoch_state['epoch'] = epoch
    return metrics

segmentation_training_module.train_one_epoch = live_train_one_epoch
segmentation_training_module.evaluate_loss = live_evaluate_loss
try:
    training_report = train(
        PROCESSED / 'train',
        PROCESSED / 'val',
        checkpoint_path,
        config=training_config,
        device='cuda',
        metadata=checkpoint_metadata,
    )
except Exception:
    if wandb_run is not None:
        wandb_run.finish(exit_code=1)
    raise
finally:
    segmentation_training_module.train_one_epoch = original_train_one_epoch
    segmentation_training_module.evaluate_loss = original_evaluate_loss
if not checkpoint_path.is_file():
    raise RuntimeError('Training completed without writing a checkpoint')
checkpoint_sha256 = sha256_file(checkpoint_path)
if wandb_run is not None:
    try:
        best_epoch = min(
            training_report['history'],
            key=lambda record: float(record['validation']['loss']),
        )['epoch']
        wandb_run.summary.update({
            'best_validation_loss': float(training_report['best_validation_loss']),
            'best_epoch': int(best_epoch) + 1,
            'checkpoint_sha256': checkpoint_sha256,
            'checkpoint_path': str(checkpoint_path),
        })
        if WANDB_LOG_CHECKPOINT_ARTIFACT:
            model_artifact = wandb.Artifact(
                f"segformer-b0-{CFG['run_mode']}",
                type='model',
                metadata={
                    'checkpoint_sha256': checkpoint_sha256,
                    'source_revision': source_revision,
                    'split_lineage_id': split_lineage['lineage_id'],
                },
            )
            model_artifact.add_file(str(checkpoint_path), name=checkpoint_path.name)
            model_artifact.add_file(
                str(checkpoint_path.with_suffix('.json')),
                name=checkpoint_path.with_suffix('.json').name,
            )
            wandb_run.log_artifact(model_artifact)
    except Exception as exc:
        wandb_metadata['logging_error'] = repr(exc)
        print('W&B logging failed after training; continuing with local artifacts:', exc)
    finally:
        wandb_run.finish()
drive_checkpoint_path = save_to_drive(
    checkpoint_path, DRIVE_CHECKPOINTS / checkpoint_path.name
)
drive_training_report_path = save_to_drive(
    checkpoint_path.with_suffix('.json'),
    DRIVE_RESULTS / checkpoint_path.with_suffix('.json').name,
)
(RESULTS / 'training_summary.json').write_text(
    json.dumps({
        **training_report,
        'checkpoint_sha256': checkpoint_sha256,
        'drive_checkpoint_path': str(drive_checkpoint_path),
        'drive_training_report_path': str(drive_training_report_path),
        'wandb': wandb_metadata,
    }, indent=2, sort_keys=True),
    encoding='utf-8',
)
print('Checkpoint:', checkpoint_path)
print('Checkpoint SHA-256:', checkpoint_sha256)
print('Checkpoint copied to Drive:', drive_checkpoint_path)
print('Best validation loss:', training_report['best_validation_loss'])


## 11. Calibrate the pixel threshold on validation only

The evaluator sweeps candidate thresholds against validation predictions, maximizes Dice under the false-clear constraint, and writes a candidate `DecisionSpec`. The test split is not used by this cell.


In [ ]:
calibration_path = RESULTS / 'validation_calibration.json'
calibration_cache = {
    'checkpoint_sha256': checkpoint_sha256,
    'raw_manifest_id': raw_manifest['raw_manifest_id'],
    'split_lineage_id': split_lineage['lineage_id'],
    'max_false_clear_rate': CFG['max_false_clear_rate'],
    'threshold_start_bp': CFG['threshold_start_bp'],
    'threshold_stop_bp': CFG['threshold_stop_bp'],
    'threshold_step_bp': CFG['threshold_step_bp'],
    'bootstrap_samples': CFG['bootstrap_samples'],
    'bootstrap_seed': CFG['seed'],
}
if not json_cache_matches(calibration_path, calibration_cache):
    restore_drive_artifact(
        calibration_path,
        DRIVE_RESULTS / calibration_path.name,
        DRIVE_DELIVERABLES / calibration_path.name,
        validator=lambda path: json_cache_matches(path, calibration_cache),
    )
if not json_cache_matches(calibration_path, calibration_cache):
    run_command(
        'Calibrate pixel threshold on validation',
        [
            sys.executable, PROJECT / 'src/eval_segmentation.py',
            '--test-dir', PROCESSED / 'val',
            '--model-path', checkpoint_path,
            '--dataset-role', 'validation',
            '--select-threshold',
            '--max-false-clear-rate', str(CFG['max_false_clear_rate']),
            '--threshold-start-bp', str(CFG['threshold_start_bp']),
            '--threshold-stop-bp', str(CFG['threshold_stop_bp']),
            '--threshold-step-bp', str(CFG['threshold_step_bp']),
            '--bootstrap-samples', str(CFG['bootstrap_samples']),
            '--bootstrap-seed', str(CFG['seed']),
            '--device', 'cuda',
            '--output', calibration_path,
        ],
        cwd=RUN,
    )
calibration_report = json.loads(calibration_path.read_text(encoding='utf-8'))
if calibration_report.get('_cube_nano_cache') != calibration_cache:
    calibration_report['_cube_nano_cache'] = calibration_cache
    calibration_path.write_text(json.dumps(calibration_report, indent=2, sort_keys=True), encoding='utf-8')
locked_threshold_bp = int(calibration_report['threshold_bp'])
candidate_decision_spec = dict(segformer_manifest['decision_spec'])
candidate_decision_spec.update({
    'pixel_cloud_probability_threshold_bp': locked_threshold_bp,
    'false_clear_constraint_bp': int(round(CFG['max_false_clear_rate'] * 10000)),
    'calibration_id': f'95cloud-validation-{checkpoint_sha256[:12]}',
    'calibration_report_sha256': sha256_file(calibration_path),
    'dataset_role': 'validation',
    'split_lineage_id': split_lineage['lineage_id'],
})
locked_decision_path = CONTRACTS / 'candidate_decision_spec.json'
locked_decision_path.write_text(
    json.dumps(candidate_decision_spec, indent=2, sort_keys=True), encoding='utf-8'
)
print('Locked validation threshold (bp):', locked_threshold_bp)
print('Candidate decision spec:', locked_decision_path)


## 12. Run the frozen test evaluation once

This uses the locked validation threshold without a test sweep. It reports micro/macro scene metrics, boundary F1, coverage errors, and deterministic scene bootstrap intervals.


In [ ]:
test_report_path = RESULTS / 'test_evaluation.json'
test_cache = {
    'checkpoint_sha256': checkpoint_sha256,
    'raw_manifest_id': raw_manifest['raw_manifest_id'],
    'split_lineage_id': split_lineage['lineage_id'],
    'threshold_bp': locked_threshold_bp,
    'bootstrap_samples': CFG['bootstrap_samples'],
    'bootstrap_seed': CFG['seed'],
}
if not json_cache_matches(test_report_path, test_cache):
    restore_drive_artifact(
        test_report_path,
        DRIVE_RESULTS / test_report_path.name,
        DRIVE_DELIVERABLES / test_report_path.name,
        validator=lambda path: json_cache_matches(path, test_cache),
    )
if not json_cache_matches(test_report_path, test_cache):
    run_command(
        'Evaluate frozen test split',
        [
            sys.executable, PROJECT / 'src/eval_segmentation.py',
            '--test-dir', PROCESSED / 'test',
            '--model-path', checkpoint_path,
            '--dataset-role', 'test',
            '--threshold-bp', str(locked_threshold_bp),
            '--bootstrap-samples', str(CFG['bootstrap_samples']),
            '--bootstrap-seed', str(CFG['seed']),
            '--device', 'cuda',
            '--output', test_report_path,
        ],
        cwd=RUN,
    )
test_report = json.loads(test_report_path.read_text(encoding='utf-8'))
if test_report.get('_cube_nano_cache') != test_cache:
    test_report['_cube_nano_cache'] = test_cache
    test_report_path.write_text(json.dumps(test_report, indent=2, sort_keys=True), encoding='utf-8')
metrics = test_report['metrics']
coverage = test_report['coverage_metrics']
quality = acceptance_profile['quality']
def check_at_least(name, actual, expected):
    return {'actual': actual, 'expected': expected, 'passed': actual is not None and actual >= expected}

def check_at_most(name, actual, expected):
    return {'actual': actual, 'expected': expected, 'passed': actual is not None and actual <= expected}

quality_gates = {
    'cloud_iou': check_at_least('cloud_iou', metrics['cloud_iou'], quality['min_cloud_iou']),
    'cloud_dice': check_at_least('cloud_dice', metrics['cloud_dice'], quality['min_cloud_dice']),
    'cloud_recall': check_at_least('cloud_recall', metrics['cloud_recall'], quality['min_cloud_recall']),
    'false_clear_rate': check_at_most('false_clear_rate', metrics['false_clear_rate'], quality['max_false_clear_rate']),
    'boundary_f1': check_at_least('boundary_f1', metrics['boundary_f1'], quality['min_boundary_f1']),
    'coverage_mae_bp': check_at_most('coverage_mae_bp', coverage['coverage_mae_bp'], quality['max_coverage_mae_bp']),
    'coverage_p95_abs_error_bp': check_at_most(
        'coverage_p95_abs_error_bp', coverage['coverage_p95_abs_error_bp'], quality['max_coverage_p95_abs_error_bp']
    ),
    'valid_pixel_ratio': check_at_least(
        'valid_pixel_ratio', test_report['valid_pixel_ratio'], acceptance_profile['runtime']['min_valid_pixel_ratio']
    ),
}
quality_gate_report = {
    'run_mode': CFG['run_mode'],
    'checkpoint_sha256': checkpoint_sha256,
    'split_lineage_id': split_lineage['lineage_id'],
    'threshold_bp': locked_threshold_bp,
    'all_quality_gates_passed': all(item['passed'] for item in quality_gates.values()),
    'gates': quality_gates,
    'release_status': 'not_a_release',
    'release_blockers': [
        'This notebook does not create a pinned pretrained artifact.',
        'This notebook runs one training seed; a final candidate requires three seeds or a documented resource exception.',
        'The model manifest, calibration binding, target benchmark, and TensorRT parity are not mutated by this run.',
        'Release promotion remains subject to the integration-plan gates.',
    ],
}
quality_gate_path = RESULTS / 'quality_gate_report.json'
quality_gate_path.write_text(
    json.dumps(quality_gate_report, indent=2, sort_keys=True), encoding='utf-8'
)
print(json.dumps(quality_gate_report, indent=2, sort_keys=True))


### Test-set inference smoke check

Run the standalone tiled inference entry point on one real image from the processed `test` split and compare its predicted mask with that image's ground truth. This cell can run without the training and calibration cells when a matching checkpoint/calibration report is already cached on Drive. The preceding evaluation cell remains the canonical aggregate report.


In [ ]:
import tifffile
import torch
from src.eval_segmentation import segmentation_metrics

checkpoint_name = f"segformer_b0_rgb_{CFG['run_mode']}.pth"
default_checkpoint_path = RUN / 'checkpoints' / checkpoint_name
checkpoint_path = Path(globals().get('checkpoint_path', default_checkpoint_path))
if not checkpoint_path.is_file():
    checkpoint_path = default_checkpoint_path
if not checkpoint_path.is_file():
    restore_drive_artifact(
        checkpoint_path,
        DRIVE_CHECKPOINTS / checkpoint_name,
    )
if not checkpoint_path.is_file():
    raise FileNotFoundError(
        f'No trained checkpoint found at {checkpoint_path} or Drive cache '
        f'{DRIVE_CHECKPOINTS / checkpoint_name}'
    )
checkpoint_sha256 = sha256_file(checkpoint_path)

if 'locked_threshold_bp' not in globals():
    threshold_source = None
    report_candidates = (
        RESULTS / 'validation_calibration.json',
        DRIVE_RESULTS / 'validation_calibration.json',
        DRIVE_DELIVERABLES / 'validation_calibration.json',
        RESULTS / 'test_evaluation.json',
        DRIVE_RESULTS / 'test_evaluation.json',
        DRIVE_DELIVERABLES / 'test_evaluation.json',
    )
    for report_path in dict.fromkeys(Path(path) for path in report_candidates):
        if not report_path.is_file():
            continue
        try:
            candidate_report = json.loads(report_path.read_text(encoding='utf-8'))
        except (OSError, json.JSONDecodeError):
            continue
        cache = candidate_report.get('_cube_nano_cache', {})
        if cache.get('checkpoint_sha256') != checkpoint_sha256:
            continue
        candidate_threshold = candidate_report.get('threshold_bp', cache.get('threshold_bp'))
        if candidate_threshold is None or not 0 <= int(candidate_threshold) <= 10000:
            continue
        locked_threshold_bp = int(candidate_threshold)
        threshold_source = report_path
        break
    if threshold_source is not None:
        print('Reused calibrated threshold (bp):', locked_threshold_bp)
        print('Threshold source:', threshold_source)
    else:
        locked_threshold_bp = int(segformer_manifest['decision_spec']['pixel_cloud_probability_threshold_bp'])
        print(
            'No calibration/evaluation report matched the checkpoint; using manifest threshold (bp):',
            locked_threshold_bp,
        )

test_image_paths = sorted(
    path
    for label in ('cloud', 'clear')
    for path in (PROCESSED / 'test' / label).glob('*.npy')
)
if not test_image_paths:
    raise RuntimeError('No processed images found in the test split')

test_image_path = test_image_paths[0]
test_mask_path = PROCESSED / 'test' / 'masks' / test_image_path.name
test_validity_path = PROCESSED / 'test' / 'validity' / test_image_path.name
inference_dir = ARTIFACTS / 'test_inference'
inference_dir.mkdir(parents=True, exist_ok=True)
inference_mask_path = inference_dir / f'{test_image_path.stem}_predicted_mask.tif'
inference_probability_path = inference_dir / f'{test_image_path.stem}_cloud_probability.tif'
inference_device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Inference device:', inference_device)

run_command(
    'Standalone SegFormer inference on a test-set image',
    [
        sys.executable, PROJECT / 'src/inference_segformer.py',
        '--image', test_image_path,
        '--checkpoint', checkpoint_path,
        '--output-mask', inference_mask_path,
        '--output-probability', inference_probability_path,
        '--tile-size', '384',
        '--batch-size', '1',
        '--threshold', str(float(locked_threshold_bp) / 10000.0),
        '--device', inference_device,
        '--input-scale', '65535',
    ],
    cwd=RUN,
)

predicted_mask = tifffile.imread(inference_mask_path) == 255
target_mask = np.load(test_mask_path, allow_pickle=False)
validity_mask = (
    np.load(test_validity_path, allow_pickle=False)
    if test_validity_path.is_file()
    else np.ones(target_mask.shape, dtype=np.uint8)
)
if predicted_mask.shape != target_mask.shape or target_mask.shape != validity_mask.shape:
    raise ValueError(
        f'Inference/ground-truth shape mismatch: {predicted_mask.shape}, '
        f'{target_mask.shape}, {validity_mask.shape}'
    )
sample_metrics = segmentation_metrics(
    predicted_mask.astype(np.float32),
    target_mask,
    validity_mask,
    locked_threshold_bp,
)
sample_inference_report = {
    'image': str(test_image_path),
    'ground_truth': str(test_mask_path),
    'predicted_mask': str(inference_mask_path),
    'cloud_probability': str(inference_probability_path),
    'threshold_bp': int(locked_threshold_bp),
    'metrics': sample_metrics,
}
sample_inference_report_path = RESULTS / 'test_inference_sample.json'
sample_inference_report_path.write_text(
    json.dumps(sample_inference_report, indent=2, sort_keys=True),
    encoding='utf-8',
)
print(json.dumps(sample_inference_report, indent=2, sort_keys=True))


## 13. Export the fixed runtime graph and check PyTorch/ONNX parity

The runtime graph stays fixed at batch 1 and `256 x 256`. A padded/cropped normalized validation tile is stored as a golden input together with raw PyTorch and ONNX logits. TensorRT parity must still run on the pinned target outside Colab.


In [ ]:
onnx_path = ARTIFACTS / f"segformer_b0_rgb_{CFG['run_mode']}.onnx"
onnx_contract_path = onnx_path.with_suffix('.onnx.json')
def onnx_cache_matches(path):
    try:
        contract = json.loads(Path(path).read_text(encoding='utf-8'))
        return contract.get('checkpoint_sha256') == checkpoint_sha256
    except (OSError, json.JSONDecodeError):
        return False
onnx_ready = onnx_path.is_file() and onnx_contract_path.is_file() and onnx_cache_matches(onnx_contract_path)
if not onnx_ready:
    restore_drive_artifact(
        onnx_contract_path,
        DRIVE_RESULTS / onnx_contract_path.name,
        DRIVE_DELIVERABLES / onnx_contract_path.name,
        validator=onnx_cache_matches,
    )
    if onnx_contract_path.is_file() and onnx_cache_matches(onnx_contract_path):
        restore_drive_artifact(
            onnx_path,
            DRIVE_RESULTS / onnx_path.name,
            DRIVE_DELIVERABLES / onnx_path.name,
        )
    onnx_ready = onnx_path.is_file() and onnx_contract_path.is_file() and onnx_cache_matches(onnx_contract_path)
if not onnx_ready:
    run_command(
        'Export fixed-shape ONNX',
        [
            sys.executable, PROJECT / 'src/export_segformer_onnx.py',
            '--checkpoint', checkpoint_path,
            '--output', onnx_path,
        ],
        cwd=RUN,
    )

import onnxruntime as ort
from src.models.segformer_b0 import get_segformer_b0

golden_dataset = SegmentationDataset(PROCESSED / 'val', is_train=False, preserve_native_size=True)
golden_sample = golden_dataset[0]['image'].cpu().numpy()
golden_input = np.zeros((1, 3, 256, 256), dtype=np.float32)
copy_height = min(256, golden_sample.shape[1])
copy_width = min(256, golden_sample.shape[2])
golden_input[0, :, :copy_height, :copy_width] = golden_sample[:, :copy_height, :copy_width]

model = get_segformer_b0().eval()
checkpoint = torch.load(checkpoint_path, map_location='cpu')
model.load_state_dict(checkpoint['model_state_dict'])
with torch.inference_mode():
    pytorch_logits = model(torch.from_numpy(golden_input)).cpu().numpy()
session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
onnx_logits = session.run(['logits'], {'input': golden_input})[0]
if pytorch_logits.shape != (1, 2, 64, 64) or onnx_logits.shape != (1, 2, 64, 64):
    raise ValueError(f'Unexpected fixed-shape logits: {pytorch_logits.shape}, {onnx_logits.shape}')
difference = np.abs(pytorch_logits.astype(np.float32) - onnx_logits.astype(np.float32))
onnx_tolerance = float(acceptance_profile['parity']['pytorch_onnx'])
parity_report = {
    'input_shape': list(golden_input.shape),
    'output_shape': list(pytorch_logits.shape),
    'max_abs_difference': float(difference.max()),
    'mean_abs_difference': float(difference.mean()),
    'tolerance': onnx_tolerance,
    'passed': bool(np.allclose(pytorch_logits, onnx_logits, rtol=0.0, atol=onnx_tolerance)),
    'tensorrt': {
        'status': 'not_run',
        'reason': 'TensorRT parity must run on the pinned deployment target, not on a generic Colab GPU.',
    },
}
np.save(ARTIFACTS / 'golden_input.npy', golden_input)
np.save(ARTIFACTS / 'golden_pytorch_logits.npy', pytorch_logits)
np.save(ARTIFACTS / 'golden_onnx_logits.npy', onnx_logits)
parity_path = RESULTS / 'pytorch_onnx_parity.json'
parity_path.write_text(json.dumps(parity_report, indent=2, sort_keys=True), encoding='utf-8')
if not parity_report['passed']:
    raise AssertionError(f'PyTorch/ONNX parity failed: {parity_report}')
print(json.dumps(parity_report, indent=2, sort_keys=True))


## 14. Package the evidence bundle

Only model/reports/contracts/golden vectors are zipped. The raw and processed datasets stay out of the download archive.


In [ ]:
environment_report = {
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'cuda_available': bool(torch.cuda.is_available()),
    'packages': {
        name: importlib.metadata.version(name)
        for name in ('numpy', 'torch', 'tifffile', 'onnx', 'onnxruntime', 'PyYAML', 'kaggle')
    },
}
if torch.cuda.is_available():
    environment_report['gpu_name'] = torch.cuda.get_device_name(0)
    try:
        environment_report['nvidia_smi'] = subprocess.check_output(['nvidia-smi', '-q'], text=True)
    except (FileNotFoundError, subprocess.CalledProcessError):
        environment_report['nvidia_smi'] = 'unavailable'
environment_path = RESULTS / 'environment.json'
environment_path.write_text(json.dumps(environment_report, indent=2, sort_keys=True), encoding='utf-8')

bundle_files = [
    checkpoint_path,
    checkpoint_path.with_suffix('.json'),
    RESULTS / 'source_provenance.json',
    RESULTS / 'raw_dataset_audit.json',
    RESULTS / 'dataset_validation.json',
    RESULTS / 'train_rgb_statistics.json',
    RESULTS / 'training_summary.json',
    calibration_path,
    test_report_path,
    quality_gate_path,
    parity_path,
    environment_path,
    locked_decision_path,
    split_manifest_path,
    split_lineage_path,
    onnx_path,
    onnx_path.with_suffix('.onnx.json'),
    ARTIFACTS / 'golden_input.npy',
    ARTIFACTS / 'golden_pytorch_logits.npy',
    ARTIFACTS / 'golden_onnx_logits.npy',
    inference_mask_path,
    inference_probability_path,
    sample_inference_report_path,
]
for source in bundle_files:
    source = Path(source)
    if not source.is_file():
        raise FileNotFoundError(f'Expected artifact is missing: {source}')
    copy_if_absent_or_same(source, DELIVERABLES / source.name)

archive_base = CONTENT / 'segformer_95cloud_evidence_bundle'
archive_target = archive_base.with_suffix('.zip')
drive_archive_candidate = DRIVE_ROOT / archive_target.name
if archive_target.is_file():
    archive_path = archive_target
    print('Reusing existing local evidence bundle:', archive_path)
elif drive_archive_candidate.is_file():
    archive_path = copy_if_absent_or_same(drive_archive_candidate, archive_target)
    print('Reusing existing Drive evidence bundle:', archive_path)
else:
    archive_path = shutil.make_archive(str(archive_base), 'zip', RUN, 'deliverables')
for source in DELIVERABLES.iterdir():
    if source.is_file():
        save_to_drive(source, DRIVE_DELIVERABLES / source.name)
drive_archive_path = save_to_drive(archive_path, DRIVE_ROOT / Path(archive_path).name)
print('Evidence bundle:', archive_path)
print('Evidence bundle copied to Drive:', drive_archive_path)

from google.colab import files
files.download(archive_path)

if CFG['cleanup_local_after_bundle']:
    remove_content_path(LOCAL_PROCESSED)
    disk_report('After local processed-data cleanup')
